# Tax-Loss Harvesting Strategy

### Wealth Management Portfolio Optimization — Banking-AI

This notebook explains each step in plain language so new learners can follow:

## Step 0 — Notebook Header

**Prerequisites:** Basic Python (`pandas`, `numpy`, `matplotlib`), and basic understanding of wealth management and portfolio optimisation terminology.

**What You Will Learn:**

After completing this notebook, you will be able to:
- Describe the information-extraction task and its relevance to wealth management and portfolio optimisation.
- Implement a rule-based extraction pipeline and evaluate accuracy on synthetic examples.
- Assess limitations of rule-based extraction and when ML-based approaches would be more appropriate.

**Estimated time:** 35–45 minutes

**Collection context:** Portfolio optimisation, asset allocation, risk management, and investment strategy for wealth management.

## Step 1 — Banking Problem Introduction

**Why this notebook matters:** It's not about what you make; it's about what you *keep*. Tax-Loss Harvesting (TLH) is a technique where you sell an investment that has lost value to 'offset' taxes on investments that have gained value. This is a core value proposition for modern 'Robo-Advisors'.

**Input data used:** Daily prices of 10 synthetic stocks, tax rates (short-term vs long-term).

**What we calculate:** Net tax liability and Tax Alpha.

**Math method used:** Iterative logic and conditional accounting.

**Learning goal:** Understand how algorithmic trading can automate tax efficiency.

**Primary stakeholders:** wealth advisors, portfolio managers, investment committee members, and financial planners.

**Comprehension check:** Before looking at the data, write one sentence describing the core decision this model supports and who benefits from getting it right.

**Domain framing note:** This notebook uses synthetic data for educational purposes. It demonstrates decision-support prototyping, not production-ready deployment.

## Step 2 — Environment Setup

Import libraries and set deterministic seeds for reproducibility.

In [ ]:
# Requires: numpy>=1.24, pandas>=2.0, scikit-learn>=1.3, matplotlib>=3.7, seaborn>=0.13

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)
plt.rcParams["figure.figsize"] = (10, 6)

print("Environment ready.")


## Step 3 — Data Creation & Context

Simulate Stock Portfolio

We create 10 stocks. Some will go up (winners), some will go down (losers).

In [ ]:
n_stocks = 10
days = 252
initial_value = 10000 # $10k in each stock

# Generate random daily returns
returns = RNG.normal(0.0005, 0.02, (days, n_stocks))
price_paths = initial_value * np.exp(np.cumsum(returns, axis=0))

df_prices = pd.DataFrame(price_paths, columns=[f'Stock_{i}' for i in range(n_stocks)])
df_prices.plot(legend=False, alpha=0.7, title='Portfolio Stock Price Paths')
plt.tight_layout()
plt.show()
plt.close()

## Step 4 — Exploratory Data Analysis

For this workflow, primary exploration happens through the operational visualisations in later steps.

## Step 5 — Feature Engineering

Suppose at the end of the year, we want to realize losses to offset $5,000 in capital gains we made elsewhere.

In [ ]:
final_values = df_prices.iloc[-1]
gains_losses = final_values - initial_value

df_status = pd.DataFrame({
    'Initial': [initial_value] * n_stocks,
    'Current': final_values,
    'Gain/Loss': gains_losses
})

print("Portfolio Year-End Status:")
print(df_status.sort_values('Gain/Loss'))

## Step 6 — Baseline Establishment

A baseline sets the floor that any useful model must beat. Without it, performance numbers have no context.

In [ ]:
# --- Baseline: manual extraction (no automation) ---
print("Baseline: without automation, each document must be read and keyed manually.")
print("Any automated approach that achieves >80% field accuracy saves significant time.")

## Step 7 — Model Training & Evaluation

Train the model and compare performance against the baseline.

**Prediction prompt:** Before viewing the results, what performance do you expect and why?

In [ ]:
tax_rate = 0.30 # Assume 30% tax on gains
external_gains = 5000

# Scenario A: Do Nothing
tax_a = external_gains * tax_rate

# Scenario B: Harvest Losses
total_harvestable_loss = abs(df_status[df_status['Gain/Loss'] < 0]['Gain/Loss'].sum())
net_gain_b = max(0, external_gains - total_harvestable_loss)
tax_b = net_gain_b * tax_rate

tax_savings = tax_a - tax_b
tax_alpha = tax_savings / (n_stocks * initial_value)

print(f"Initial Tax Bill: ${tax_a:,.2f}")
print(f"New Tax Bill (after harvesting): ${tax_b:,.2f}")
print(f"Total Savings: ${tax_savings:,.2f}")
print(f"Tax Alpha (Extra Return): {tax_alpha:.2%}")

In [ ]:
labels = ['Tax Blind Portfolio', 'Tax Aware Portfolio']
values = [external_gains * (1-tax_rate), external_gains - tax_b]

plt.figure(figsize=(8, 5))
sns.barplot(x=labels, y=values)
plt.title('After-Tax Gains Comparison')
plt.ylabel('Net Cash ($)')
plt.tight_layout()
plt.show()
plt.close()

**Alt-text:** Bar chart titled "After-Tax Gains Comparison". The chart ranks features or categories by magnitude to highlight dominant factors.

## Step 8 — Interpretability & Explainability

Interpret the model in domain terms. A banking professional should be able to assess whether the model's reasoning is credible.

1. **Hidden Returns:** Even if the market is flat, a tax-aware algorithm can generate 1-2% of 'extra' return just by managing the tax bill.
2. **Wash Sale Rule:** In the real world, you can't buy the same stock back for 30 days. Algorithms handle this by buying a similar (but not identical) ETF to stay invested while the loss is 'locked in'.
3. **Robo-Advising:** This is why platforms like Wealthfront or Betterment are popular—they do this every day automatically, which is too tedious for a human to do manually.

## Step 9 — Operational Application

Demonstrate how the model output feeds into a real banking workflow decision.

In [ ]:
# --- Scenario: process a new document ---
print("Operational demonstration — field extraction:\n")
new_doc = "Invoice #2055. Vendor: Wayne Enterprises. Date: 2024-06-15. Total: $8750.00"
result = extract_info(new_doc)
print(f"  Raw: {new_doc}")
print(f"  Vendor: {result[0]}  Date: {result[1]}  Amount: ${result[2]:,.2f}")
print("\nExtracted fields would auto-populate the accounts-payable system.")

## Step 10 — Summary & Reflection

**What you accomplished:** You built an end-to-end wealth management and portfolio optimisation workflow — from problem framing through data creation, baseline comparison, model training, evaluation, and operational demonstration.

**Limitations:**

- The dataset is synthetic and cannot capture the full complexity of real banking data.
- Model performance on synthetic data does not guarantee equivalent performance in production.
- This notebook is an educational prototype. Real deployment requires validated data, regulatory review, and ongoing monitoring.

**Ethical reflection:**

- Optimisation models may suggest portfolios unsuitable for clients with different risk tolerances or time horizons.
- Model assumptions about return distributions may not hold during market crises.
- Fiduciary duty requires that model outputs support, not replace, professional judgement.

**Reflection questions:**

1. What additional data sources or features would you need before trusting this model in a live banking environment?
2. How would the relative cost of false positives versus false negatives change the metric you optimise for?
3. How could you adapt this workflow to a related problem in wealth management and portfolio optimisation?

## References

- Scikit-learn documentation: [https://scikit-learn.org/stable/](https://scikit-learn.org/stable/)
- Project quality baseline: `QUALITY_STANDARD.md`
- Domain context drawn from public banking and financial services literature.